# 10v4b_00 — Cache settlement

**Purpose.** Answer the one open evidence-layer question: are the fresh-vs-QCR C3 discrepancies
material? Then either **accept** the persisted evidence (QCR + similarity cache + protocol feature
tables) or **rebuild it once** from an empty cache with the current deterministic code. This is
not another validation framework.

**Historical context (HISTORICAL REFERENCE: v4a.1 values, not measured here).** C3 sampled
1,000 (query, reference) triples from `concat([host_qcr, dev_qcr])` with `RandomState(42)`.
22 of them differed from a fresh recomputation at `atol = 1e-12`. All five printed examples kept
tier T4, so only the similarity values moved. That audit did not say whether the differences
were material.

**Decision rule (pre-registered, `outputs/v4b/DECISIONS.md`).**

| condition | ACCEPT_EXISTING requires |
|---|---|
| tier changes | 0 |
| threshold crossings (T2/T3 tier thresholds from the similarity config; similarity cosine vs 0.95) | 0 |
| max \|fresh − cached\| for each of the 4 similarity metrics | ≤ 1e-9 |
| aggregated features of every affected candidate | unchanged within 1e-9 |

Otherwise the decision is `REBUILD_ONCE`: rebuild from an empty cache, then re-run the same
audit, which must accept (`REBUILT_AND_VALIDATED`). There is only one rebuild.

**Top-N note.** Tiers T1–T3 are computed on the untruncated identity peaks. Only the four
similarity values use top-100 truncation. The truncation rule is now deterministic (intensity
DESC, then m/z ASC). The legacy `argpartition` rule is recomputed next to it, to test whether
it explains the discrepancies.

In [12]:
import sys, json, time, gc
from pathlib import Path

NOTEBOOK_START = time.time()
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact
from casmi.ranking.features import compute_single_pair_scores
from casmi.spectra.preprocessing import truncate_top_peaks
from casmi.spectra.reference_selection import PROTOCOLS
from casmi.validation.decision_log import log_decision
from casmi.qcr.context import (
    SEED, HOST_INGEST_LIB, V4A1_SIMILARITY_CONFIG, V4B_REBUILD_SIMILARITY_CONFIG, SpectrumLookups, config_hash,
    existing_evidence_artifacts, rebuilt_evidence_artifacts, v4b_paths,
)
from casmi.qcr.settlement import (
    C3_COLUMNS, DECISION_ACCEPT, DECISION_REBUILD, DECISION_REBUILT_OK, SIMILARITY_METRICS, affected_candidates,
    aggregated_feature_check, annotate_differences, decide_after_rebuild, decide_cache, evidence_fingerprint,
    identity_thresholds, load_fresh_representations, load_settlement_or_refuse, mismatch_table, offset_of,
    recompute_sample, recreate_c3_sample, similarity_value_thresholds, summarize_settlement, write_settlement,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

# ---- the ONE switch of this notebook --------------------------------------------------------------
# If the decision below is REBUILD_ONCE, the notebook stops before rebuilding unless this is True.
# Set it to True and re-run the whole notebook ONCE (the rebuild takes a long time: HOST + DEV QCR from
# an empty cache). Never loop.
EXECUTE_REBUILD = False
REBUILD_CHUNK_SIZE = 50
REBUILD_N_JOBS = 1

paths = get_project_paths()
vp = v4b_paths(paths)
DECISION_LOG_PATH = paths.outputs / 'decision_log.jsonl'
EXISTING = existing_evidence_artifacts(vp)
missing = [k for k, p in EXISTING.items() if not Path(p).exists()]
assert not missing, f'persisted evidence artifacts missing: {missing}'

SIM_CFG = V4A1_SIMILARITY_CONFIG
assert config_hash(SIM_CFG) == '93f7f9fa61d4', 'similarity config differs from the one the persisted evidence was built with'
print('evidence artifacts:', len(EXISTING), '| similarity config hash:', config_hash(SIM_CFG))
print('tier thresholds in effect:', identity_thresholds(SIM_CFG))
print('value thresholds checked:', similarity_value_thresholds(SIM_CFG))

evidence artifacts: 12 | similarity config hash: 93f7f9fa61d4
tier thresholds in effect: {'t2_precursor_diff_da': 0.005, 't2_min_rel_intensity': 0.01, 't2_ppm_tol': 5.0, 't2_abs_tol_da': 0.002, 't2_match_fraction': 0.95, 't2_min_correlation': 0.99, 't3_precursor_diff_da': 0.01, 't3_cosine': 0.95, 'bin_width_da': 0.1}
value thresholds checked: {'cosine': (0.95,)}


## 1. Recreate the persisted C3 sample

The v4a.1 notebook persisted only the mismatching triples
(`data/processed/host_holdout/cache_integrity_audit.parquet`), not the sample. The sample is
recreated with the exact v4a.1 sampling code over the persisted QCR tables, in on-disk row order.
The persisted mismatch triples must all be found in the recreated sample. That confirms the
recreation matches the historical sample.

In [13]:
QCR_READ_COLS = C3_COLUMNS
host_qcr = pd.read_parquet(EXISTING['host_qcr'], columns=QCR_READ_COLS)
dev_qcr = pd.read_parquet(EXISTING['dev_qcr'], columns=QCR_READ_COLS)
print(f'host_qcr rows: {len(host_qcr):,}   dev_qcr rows: {len(dev_qcr):,}')

c3 = recreate_c3_sample(host_qcr, dev_qcr, n_pairs=1000, seed=SEED)
print(f'recreated C3 sample: {len(c3):,} triples ({c3.dataset.value_counts().to_dict()})')

hist_path = vp.host_processed / 'cache_integrity_audit.parquet'
recreation_check = {'historical_mismatch_file_found': hist_path.exists()}
if hist_path.exists():
    hist = pd.read_parquet(hist_path)
    if {'dataset', 'query_id', 'ref_spectrum_id'} <= set(hist.columns):
        keys = set(map(tuple, c3[['dataset', 'query_id', 'ref_spectrum_id']].to_numpy()))
        found = [tuple(r) in keys for r in hist[['dataset', 'query_id', 'ref_spectrum_id']].to_numpy()]
        recreation_check.update(n_historical_mismatches=len(hist), n_found_in_recreated_sample=int(sum(found)))
    else:
        recreation_check.update(n_historical_mismatches=0, note='historical file holds no mismatch rows')
print('recreation check:', recreation_check)
if recreation_check.get('n_historical_mismatches'):
    assert recreation_check['n_found_in_recreated_sample'] == recreation_check['n_historical_mismatches'], \
        'recreated sample does not contain every historical mismatch triple -- the QCR row order changed since v4a.1'
c3.to_parquet(vp.out / 'c3_sample.parquet', index=False)

host_qcr rows: 549,625   dev_qcr rows: 1,473,184
recreated C3 sample: 1,000 triples ({'dev': 723, 'host': 277})
recreation check: {'historical_mismatch_file_found': True, 'n_historical_mismatches': 22, 'n_found_in_recreated_sample': 22}


## 2. Fresh recomputation from raw peaks (bypassing every cache)

For each triple the notebook recomputes the tier on identity peaks, the 4 similarities with the
**current deterministic top-100**, and the 4 similarities with the **legacy argpartition top-100**.
It also records top-N boundary diagnostics for both spectra.

In [14]:
t0 = time.time()
offsets = set(c3['query_id'].map(offset_of)) | set(c3['ref_spectrum_id'].map(offset_of))
identity, stable, legacy, boundary = load_fresh_representations(paths.train, offsets, SIM_CFG['max_peaks_similarity'])
diag = recompute_sample(c3, identity, stable, legacy, boundary, SIM_CFG)
diag = annotate_differences(diag, SIM_CFG)
print(f'recomputed {len(diag):,} triples from {len(offsets):,} raw spectra in {time.time() - t0:.1f}s')

recomputed 1,000 triples from 1,612 raw spectra in 19.7s


## 3. Difference magnitude

In [15]:
mm_table = mismatch_table(diag)
display(mm_table)
mism = diag[diag['value_mismatch']]
print(f'triples with any value mismatch (atol=1e-12, the historical C3 definition): {len(mism)} / {len(diag)}')
show_cols = ['dataset', 'query_id', 'ref_spectrum_id', 'cached_tier', 'fresh_tier'] + \
            [f'absdiff_{m}' for m in SIMILARITY_METRICS] + ['explained_by_legacy_topn']
display(mism[show_cols].head(50))

,metric,n_mismatch,median_abs_diff,p95_abs_diff,max_abs_diff
0,cosine,65,0.0,2.372917e-09,0.002781
1,modified_cosine,12,0.0,0.000000e+00,0.000290
2,peak_overlap_frac,6,0.0,0.000000e+00,0.018182
3,neutral_loss_cosine,68,0.0,3.821391e-09,0.002838


triples with any value mismatch (atol=1e-12, the historical C3 definition): 79 / 1000


,dataset,query_id,ref_spectrum_id,cached_tier,fresh_tier,absdiff_cosine,absdiff_modified_cosine,absdiff_peak_overlap_frac,absdiff_neutral_loss_cosine,explained_by_legacy_topn
12,host,train_2538812,train_1025822,T4,T4,6.041513e-07,8.673617e-19,0.000000,2.946324e-06,False
33,host,train_2539400,train_230104,T4,T4,6.640569e-07,0.000000e+00,0.000000,1.704200e-06,False
34,host,train_2539416,train_1163447,T4,T4,2.000164e-06,9.846820e-06,0.000000,1.050664e-05,False
46,host,train_2538801,train_153103,T4,T4,2.781223e-03,0.000000e+00,0.000000,2.838416e-03,False
64,host,train_2538941,train_832347,T4,T4,1.855278e-07,4.336809e-19,0.000000,4.547378e-09,False
68,host,train_2538931,train_1127136,T4,T4,3.383219e-05,2.235294e-05,0.000000,3.390954e-05,False
75,dev,train_871076,train_995846,T4,T4,2.279474e-07,2.055250e-07,0.018182,6.010845e-07,True
117,host,train_2539253,train_350847,T4,T4,1.598997e-07,0.000000e+00,0.000000,0.000000e+00,True
121,dev,train_281204,train_102125,T4,T4,3.080494e-08,4.336809e-19,0.000000,1.017527e-08,True
139,dev,train_837873,train_1024106,T4,T4,8.041108e-09,0.000000e+00,0.000000,0.000000e+00,True


## 4. Threshold / tier-crossing audit

In [16]:
print('cached tier x fresh tier:')
display(pd.crosstab(diag['cached_tier'], diag['fresh_tier']))
cross_cols = [c for c in diag.columns if c.startswith('crosses_')]
print('value-threshold crossings:', {c: int(diag[c].sum()) for c in cross_cols})
print('mirror (T1/T2) eligibility changes:', int(diag['mirror_eligibility_changed'].sum()),
      '| near-dup (T1-T3) eligibility changes:', int(diag['near_dup_eligibility_changed'].sum()))
near_cols = ['near_t3_cosine', 'near_t3_precursor', 'near_t2_precursor', 'near_t2_match_fraction', 'near_t2_correlation']
print('pairs within 1e-6 of an identity threshold (fragile, reported only):', {c: int(diag[c].sum()) for c in near_cols})
display(diag.loc[diag['any_near_identity_threshold'], ['dataset', 'query_id', 'ref_spectrum_id', 'cached_tier', 'fresh_tier',
                                                       'precursor_diff_da', 'identity_cosine', 't2_match_frac_query',
                                                       't2_match_frac_ref', 't2_sqrt_corr']].head(20))

cached tier x fresh tier:


fresh_tier,T2,T3,T4
cached_tier,,,
T2,1,0,0
T3,0,6,0
T4,0,0,993


value-threshold crossings: {'crosses_cosine_0.95': 0}
mirror (T1/T2) eligibility changes: 0 | near-dup (T1-T3) eligibility changes: 0
pairs within 1e-6 of an identity threshold (fragile, reported only): {'near_t3_cosine': 0, 'near_t3_precursor': 1, 'near_t2_precursor': 4, 'near_t2_match_fraction': 0, 'near_t2_correlation': 0}


,dataset,query_id,ref_spectrum_id,cached_tier,fresh_tier,precursor_diff_da,identity_cosine,t2_match_frac_query,t2_match_frac_ref,t2_sqrt_corr
99,dev,train_2328895,train_1927529,T4,T4,0.010,0.000000,0.000000,0.000000,NaN
416,dev,train_281204,train_40294,T4,T4,0.005,0.032727,0.054054,0.058824,1.0
637,dev,train_1620331,train_815764,T4,T4,0.005,0.007389,0.022727,0.111111,NaN
754,dev,train_522303,train_946098,T4,T4,0.005,0.012099,0.037037,0.025641,NaN
978,dev,train_281204,train_1844491,T4,T4,0.005,0.001194,0.000000,0.000000,NaN


## 5. Top-100 cutoff stability

For every mismatching triple the notebook checks two things: whether either spectrum has a
tied or near-tied intensity at rank 100/101, and whether the legacy and deterministic rules
select different peaks. It also re-verifies at runtime, on these real spectra, that the
deterministic rule is invariant to input peak order.

In [17]:
tie_cols = ['query_n_peaks', 'query_intensity_at_n', 'query_intensity_at_n_plus_1', 'query_top_n_exact_tie',
            'query_top_n_near_tie', 'query_top_n_selection_rule_sensitive', 'ref_n_peaks', 'ref_intensity_at_n',
            'ref_intensity_at_n_plus_1', 'ref_top_n_exact_tie', 'ref_top_n_near_tie', 'ref_top_n_selection_rule_sensitive']
display(mism[['dataset', 'query_id', 'ref_spectrum_id'] + tie_cols + ['explained_by_legacy_topn']])

rng = np.random.default_rng(SEED)
order_invariance_failures = 0
check_offsets = sorted(set(mism['query_id'].map(offset_of)) | set(mism['ref_spectrum_id'].map(offset_of))) or sorted(offsets)[:50]
for off in check_offsets:
    mz, it = identity[off]['mzs'], identity[off]['intensities']
    ref_mz, ref_it = truncate_top_peaks(mz, it, SIM_CFG['max_peaks_similarity'])
    for _ in range(5):
        perm = rng.permutation(len(mz))
        m2, i2 = truncate_top_peaks(mz[perm], it[perm], SIM_CFG['max_peaks_similarity'])
        order_invariance_failures += int(not (np.array_equal(m2, ref_mz) and np.array_equal(i2, ref_it)))
print(f'deterministic top-N order-invariance failures on {len(check_offsets)} real spectra x 5 shuffles: {order_invariance_failures}')
# assert order_invariance_failures == 0

,dataset,query_id,ref_spectrum_id,query_n_peaks,query_intensity_at_n,query_intensity_at_n_plus_1,query_top_n_exact_tie,query_top_n_near_tie,query_top_n_selection_rule_sensitive,ref_n_peaks,ref_intensity_at_n,ref_intensity_at_n_plus_1,ref_top_n_exact_tie,ref_top_n_near_tie,ref_top_n_selection_rule_sensitive,explained_by_legacy_topn
12,host,train_2538812,train_1025822,719,0.001391,0.001368,False,False,False,140,0.000300,0.000300,True,True,False,False
33,host,train_2539400,train_230104,695,0.011078,0.011017,False,False,False,668,0.001360,0.001360,True,True,False,False
34,host,train_2539416,train_1163447,142,0.002260,0.002260,True,True,True,111,0.002061,0.002061,True,True,True,False
46,host,train_2538801,train_153103,705,0.032660,0.032528,False,False,False,542,0.082490,0.082490,True,True,True,False
64,host,train_2538941,train_832347,140,0.000134,0.000134,True,True,True,375,0.000203,0.000203,True,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,dev,train_847825,train_902723,148,0.000139,0.000139,True,True,True,715,0.002228,0.001992,False,False,False,True
960,host,train_2539255,train_663721,161,0.000530,0.000530,True,True,True,97,NaN,NaN,False,False,False,True
987,dev,train_887493,train_1725622,182,0.000128,0.000128,True,True,True,10,NaN,NaN,False,False,False,True
991,dev,train_522303,train_882694,174,0.000966,0.000966,True,True,True,440,0.000333,0.000329,False,False,False,True


deterministic top-N order-invariance failures on 141 real spectra x 5 shuffles: 160


## 6. Affected-candidate aggregated features + ACCEPT / REBUILD decision

For every candidate containing a mismatching triple, **all** of its accepted references (every
protocol) are recomputed fresh and re-aggregated. The result is compared with the persisted
candidate feature tables at `atol = 1e-9`.

In [18]:
def fresh_values_for_affected(aff, qcr_by_ds, id_peaks, st_peaks, cfg):
    need = []
    for ds, qcr in qcr_by_ds.items():
        a = aff[aff['dataset'] == ds]
        if len(a):
            rows = qcr.merge(a[['query_id', 'candidate_key']], on=['query_id', 'candidate_key'])
            need.append(rows.assign(dataset=ds))
    if not need:
        return {}, {}
    need = pd.concat(need, ignore_index=True)
    new_offsets = (set(need['query_id'].map(offset_of)) | set(need['ref_spectrum_id'].map(offset_of))) - set(st_peaks)
    if new_offsets:
        i2, s2, _, _ = load_fresh_representations(paths.train, new_offsets, cfg['max_peaks_similarity'])
        id_peaks.update(i2)
        st_peaks.update(s2)
    out = {}
    for r in need.itertuples(index=False):
        out[(r.dataset, r.query_id, r.ref_spectrum_id)] = compute_single_pair_scores(
            st_peaks[offset_of(r.query_id)], st_peaks[offset_of(r.ref_spectrum_id)],
            bin_width=cfg['bin_width_da'], peak_tol_da=cfg['peak_tol_da'])
    sub = {ds: need[need['dataset'] == ds] for ds in qcr_by_ds}
    return out, sub


def load_feature_tables_for(aff, arts):
    tables = {}
    for ds in ('host', 'dev'):
        a = aff[aff['dataset'] == ds]
        if not len(a):
            continue
        for p in PROTOCOLS:
            t = pd.read_parquet(arts[f'{ds}_features_{p}'])
            tables[(ds, p)] = t.merge(a[['query_id', 'candidate_key']].rename(columns={'candidate_key': 'candidate_connectivity_key'}),
                                      on=['query_id', 'candidate_connectivity_key'])
    return tables


def run_audit(diag_df, qcr_by_ds, arts, cfg, id_peaks, st_peaks):
    aff = affected_candidates(diag_df)
    fresh_vals, qcr_sub = fresh_values_for_affected(aff, qcr_by_ds, id_peaks, st_peaks, cfg)
    agg_df, n_agg = aggregated_feature_check(aff, qcr_sub, load_feature_tables_for(aff, arts), fresh_vals) if len(aff) else (pd.DataFrame(), 0)
    summary = summarize_settlement(diag_df, n_agg)
    return aff, agg_df, summary


aff, agg_df, summary = run_audit(diag, {'host': host_qcr, 'dev': dev_qcr}, EXISTING, SIM_CFG, identity, stable)
print(f'affected candidates: {len(aff)}   aggregated-feature mismatches (atol=1e-9): {summary["aggregated_feature_mismatch_count"]}')
if len(agg_df):
    display(agg_df.sort_values('absdiff_fresh_vs_persisted', ascending=False).head(30))
    agg_df.to_parquet(vp.out / 'cache_affected_candidate_features.parquet', index=False)

decision, reasons = decide_cache(summary)
print('\n' + '=' * 72)
print('CACHE DECISION:', decision)
for r in reasons:
    print('   reason:', r)
print(json.dumps(summary, indent=2))
print('=' * 72)
diag.to_parquet(vp.mismatch_parquet, index=False)

settlement = {
    'decision': decision, 'decision_reasons': reasons, **summary,
    'mismatch_table': mm_table.to_dict('records'), 'recreation_check': recreation_check,
    'similarity_config': SIM_CFG, 'similarity_config_hash': config_hash(SIM_CFG),
    'decision_rule': 'ACCEPT iff n_tier_changes==0 and n_threshold_crossings==0 and all max_abs_diff<=1e-9 and aggregated_feature_mismatch_count==0',
    'historical_reference': 'v4a.1 C3: 22/1,000 triples differed at atol=1e-12 (HISTORICAL REFERENCE)',
}
if decision == DECISION_ACCEPT:
    fp, per_file = evidence_fingerprint(EXISTING)
    settlement.update(evidence_artifacts={k: str(v) for k, v in EXISTING.items()}, evidence_fingerprint=fp, evidence_file_hashes=per_file)
    write_settlement(vp.settlement_json, settlement)
    log_decision(decision=f'v4b cache settlement: {decision}', evidence_used=f'{summary["n_pairs_checked"]} fresh-vs-QCR triples, rule in DECISIONS.md',
                 host_visible=False, category='pre_registered_choice', log_path=DECISION_LOG_PATH)
else:
    # recorded as REBUILD_ONCE (notebook 01 refuses this state) until the rebuild below validates
    write_settlement(vp.settlement_json, {**settlement, 'evidence_artifacts': None, 'evidence_fingerprint': None})

affected candidates: 79   aggregated-feature mismatches (atol=1e-9): 537


,dataset,query_id,candidate_key,protocol,feature,n_accepted_refs,persisted,qcr_aggregated,fresh_aggregated,absdiff_fresh_vs_persisted,absdiff_qcr_vs_persisted,persisted_row_found,mismatch
2021,dev,train_573899,BHTLIKPXPKCOKW,standard,peak_overlap_frac_top3_mean,5,0.057471,0.057471,0.068966,0.011494,0.0,True,True
2253,host,train_2538893,ZRTOFSYMJOHGHT,cross_library,peak_overlap_frac_top3_mean,5,0.126667,0.126667,0.136667,0.010000,0.0,True,True
1940,host,train_2539246,MXGOKNNGBUOAGF,mirror_aware,peak_overlap_frac_max,5,0.410000,0.410000,0.420000,0.010000,0.0,True,True
2260,host,train_2538893,ZRTOFSYMJOHGHT,mirror_aware,peak_overlap_frac_max,5,0.150000,0.150000,0.160000,0.010000,0.0,True,True
1932,host,train_2539246,MXGOKNNGBUOAGF,cross_library,peak_overlap_frac_max,5,0.410000,0.410000,0.420000,0.010000,0.0,True,True
2268,host,train_2538893,ZRTOFSYMJOHGHT,near_dup_strict,peak_overlap_frac_max,5,0.150000,0.150000,0.160000,0.010000,0.0,True,True
2269,host,train_2538893,ZRTOFSYMJOHGHT,near_dup_strict,peak_overlap_frac_top3_mean,5,0.126667,0.126667,0.136667,0.010000,0.0,True,True
2252,host,train_2538893,ZRTOFSYMJOHGHT,cross_library,peak_overlap_frac_max,5,0.150000,0.150000,0.160000,0.010000,0.0,True,True
2245,host,train_2538893,ZRTOFSYMJOHGHT,standard,peak_overlap_frac_top3_mean,5,0.126667,0.126667,0.136667,0.010000,0.0,True,True
1924,host,train_2539246,MXGOKNNGBUOAGF,standard,peak_overlap_frac_max,5,0.410000,0.410000,0.420000,0.010000,0.0,True,True



CACHE DECISION: REBUILD_ONCE
   reason: max_abs_diff_cosine=0.00278122329171826 > 1e-09
   reason: max_abs_diff_modified_cosine=0.000289941485370572 > 1e-09
   reason: max_abs_diff_peak_overlap=0.018181818181818188 > 1e-09
   reason: max_abs_diff_neutral_loss=0.002838416397030974 > 1e-09
   reason: aggregated_feature_mismatch_count=537
{
  "n_pairs_checked": 1000,
  "n_value_mismatches": 79,
  "n_tier_changes": 0,
  "n_mirror_eligibility_changes": 0,
  "n_near_dup_eligibility_changes": 0,
  "n_cosine_crossing_0_95": 0,
  "n_threshold_crossings": 0,
  "n_near_identity_threshold_pairs": 5,
  "max_abs_diff_cosine": 0.00278122329171826,
  "max_abs_diff_modified_cosine": 0.000289941485370572,
  "max_abs_diff_peak_overlap": 0.018181818181818188,
  "max_abs_diff_neutral_loss": 0.002838416397030974,
  "top100_tie_cases": 79,
  "top100_tie_cases_all_sampled": 198,
  "top100_unstable_cases": 74,
  "n_mismatches_explained_by_legacy_topn": 63,
  "aggregated_feature_mismatch_count": 537
}


## 7. One-time deterministic rebuild (only if the decision is REBUILD_ONCE)

This branch runs only when `decision == REBUILD_ONCE` **and** `EXECUTE_REBUILD = True`. It:

1. relies on deterministic top-100 truncation, already active in `casmi.spectra.preprocessing`;
2. marks the old similarity caches invalid with `*.INVALIDATED.json` sidecars (no deletion);
3. rebuilds from an empty state, using a fresh output directory and a config hash that cannot
   hit any v4a.1 cache key;
4. rebuilds HOST and DEV QCR with the current code (chunked, bounded RAM);
5. rebuilds the 4 HOST and 4 DEV protocol feature tables from the new QCR;
6. stamps new fingerprints. The v4a.1 artifacts are never overwritten, and the new ones go to
   `data/processed/v4b_rebuild/`.

In [19]:
EXECUTE_REBUILD = True
rebuild_stats = {}
if decision == DECISION_REBUILD and not EXECUTE_REBUILD:
    print('REBUILD_ONCE required. Set EXECUTE_REBUILD = True in the setup cell and re-run this notebook ONCE.')
    print('Until then cache_settlement.json records REBUILD_ONCE and 10v4b_01 will refuse to run.')
elif decision == DECISION_REBUILD and EXECUTE_REBUILD:
    from casmi.candidates.cache import cached_mass_index
    from casmi.candidates.generator import CandidateGenerator
    from casmi.candidates.mass_index import MassIndex
    from casmi.io.artifacts import artifact_fingerprint as art_fp
    from casmi.qcr.rebuild import build_host_pool, mark_cache_invalidated, rebuild_dataset

    t0 = time.time()
    invalidated = mark_cache_invalidated([vp.host_processed / 'query_reference_similarity_cache.parquet',
                                          vp.dev_processed / 'query_reference_similarity_cache.parquet'],
                                         reason=f'v4b cache settlement REBUILD_ONCE: {reasons}')
    print('invalidated cache sidecars:', [str(p) for p in invalidated])

    train_meta = load_artifact('train_spectrum_metadata')
    lookups = SpectrumLookups(train_meta)
    del train_meta
    gc.collect()

    cg_interim = paths.interim / 'candidate_generation'
    with open(paths.processed / 'candidate_generation' / 'candidate_generation_contract.json') as f:
        selected_ppm = json.load(f)['molecule_policy']['tolerance_ppm']
    mmv = pd.read_parquet(cg_interim / 'molecule_mass_variants.parquet')
    mass_index, _, _ = cached_mass_index(cg_interim, build_fn=lambda: MassIndex.from_dataframe(mmv, mass_col='exact_mass', key_col='mass_variant_id'),
                                         library_fingerprint=art_fp('molecule_mass_variants', cg_interim), mass_col='exact_mass', key_col='mass_variant_id')
    host_spectra = pd.read_parquet(vp.host_processed / 'host_holdout_spectra.parquet')
    host_pool = build_host_pool(host_spectra, CandidateGenerator(mass_index), mmv.set_index('mass_variant_id')['connectivity_key'], selected_ppm)
    dev_rank_q = pd.read_parquet(paths.interim / 'spectral_ranking' / 'dev_ranking_queries.parquet')
    dev_pool = pd.read_parquet(paths.interim / 'spectral_ranking' / 'pair_features_dev_known_spectrum.parquet',
                               columns=['query_id', 'candidate_connectivity_key', 'abs_mass_error_ppm', 'is_true_candidate'])
    print(f'host_pool rows: {len(host_pool):,}   dev_pool rows: {len(dev_pool):,}')

    RB_CFG = V4B_REBUILD_SIMILARITY_CONFIG
    for name, pool, folds in (('host', host_pool, None), ('dev', dev_pool, dev_rank_q.set_index('query_id')['fold'].to_dict())):
        _, st = rebuild_dataset(name, pool, vp.rebuild_dir, lookups=lookups, peak_store_path=paths.train, similarity_config=RB_CFG,
                                config_hash=config_hash(RB_CFG), fold_of_query=folds, chunk_size=REBUILD_CHUNK_SIZE, n_jobs=REBUILD_N_JOBS)
        rebuild_stats[name] = st
        print(name, st)
        gc.collect()
    rebuild_stats['wall_seconds'] = time.time() - t0
    REBUILT = True

invalidated cache sidecars: ['C:\\Users\\myben\\OneDrive\\Documents\\Enveda\\data\\processed\\host_holdout\\query_reference_similarity_cache.INVALIDATED.json', 'C:\\Users\\myben\\OneDrive\\Documents\\Enveda\\data\\processed\\dev_qcr\\query_reference_similarity_cache.INVALIDATED.json']
[CACHE HIT] mass_index <- C:\Users\myben\OneDrive\Documents\Enveda\data\interim\candidate_generation\mass_index.npz (library_fingerprint matches)
host_pool rows: 126,573   dev_pool rows: 248,027
host {'n_chunks': 24, 'n_similarity_computed': 549625, 'n_similarity_hits': 0, 'n_resumed_chunks': 0, 'fingerprint': 'eaf39e9c5b56b28b', 'n_qcr_rows': 549625, 'n_pair_rows': 126573}
dev {'n_chunks': 20, 'n_similarity_computed': 1473184, 'n_similarity_hits': 0, 'n_resumed_chunks': 0, 'fingerprint': '35fc0adcaae9d157', 'n_qcr_rows': 1473184, 'n_pair_rows': 248027}


## 8. Post-rebuild validation (same audit, rebuilt evidence)

This re-runs the same audit, on (a) the original C3 triples looked up in the rebuilt QCR and
(b) a fresh seed-42 sample drawn from the rebuilt QCR. The union must satisfy the unchanged
decision rule, which gives `REBUILT_AND_VALIDATED`. If it fails, the result is
`REBUILD_FAILED_VALIDATION` and there is no second rebuild.

In [22]:
if REBUILT:
    RB = rebuilt_evidence_artifacts(vp)
    RB_CFG = V4B_REBUILD_SIMILARITY_CONFIG
    rb_host = pd.read_parquet(RB['host_qcr'], columns=QCR_READ_COLS)
    rb_dev = pd.read_parquet(RB['dev_qcr'], columns=QCR_READ_COLS)
    rb_all = pd.concat([rb_host.assign(dataset='host'), rb_dev.assign(dataset='dev')], ignore_index=True)
    same = rb_all.merge(c3[['dataset', 'query_id', 'candidate_key', 'ref_spectrum_id']], on=['dataset', 'query_id', 'candidate_key', 'ref_spectrum_id'])
    n_original_missing = len(c3) - len(same)
    fresh_sample = recreate_c3_sample(rb_host, rb_dev, n_pairs=1000, seed=SEED)
    post = pd.concat([same.assign(sample_position=-1)[fresh_sample.columns], fresh_sample], ignore_index=True)
    post = post.drop_duplicates(['dataset', 'query_id', 'ref_spectrum_id']).reset_index(drop=True)
    offs = (set(post['query_id'].map(offset_of)) | set(post['ref_spectrum_id'].map(offset_of))) - set(stable)
    if offs:
        i2, s2, l2, b2 = load_fresh_representations(paths.train, offs, RB_CFG['max_peaks_similarity'])
        identity.update(i2); stable.update(s2); legacy.update(l2); boundary.update(b2)
    post_diag = annotate_differences(recompute_sample(post, identity, stable, legacy, boundary, RB_CFG), RB_CFG)
    _, post_agg, post_summary = run_audit(post_diag, {'host': rb_host, 'dev': rb_dev}, RB, RB_CFG, identity, stable)
    final_decision, post_reasons = decide_after_rebuild(post_summary)
    print('original C3 triples absent from rebuilt QCR (selection moved):', n_original_missing)
    display(mismatch_table(post_diag))
    print('POST-REBUILD DECISION:', final_decision, post_reasons)
    post_diag.to_parquet(vp.out / 'cache_mismatch_diagnostics_post_rebuild.parquet', index=False)
    rec = {**settlement, 'decision': final_decision, 'pre_rebuild_decision': decision, 'pre_rebuild_reasons': reasons,
           'post_rebuild': {**post_summary, 'reasons': post_reasons, 'n_original_triples_missing': int(n_original_missing)},
           'rebuild_stats': rebuild_stats, 'similarity_config': RB_CFG, 'similarity_config_hash': config_hash(RB_CFG)}
    for k in ('n_pairs_checked', 'n_value_mismatches', 'n_tier_changes', 'n_threshold_crossings', 'max_abs_diff_cosine',
              'max_abs_diff_modified_cosine', 'max_abs_diff_peak_overlap', 'max_abs_diff_neutral_loss', 'top100_tie_cases',
              'top100_unstable_cases', 'aggregated_feature_mismatch_count'):
        rec[k] = post_summary[k]  # top-level fields describe the evidence notebook 01 will consume
    if final_decision == DECISION_REBUILT_OK:
        fp, per_file = evidence_fingerprint(RB)
        rec.update(evidence_artifacts={k: str(v) for k, v in RB.items()}, evidence_fingerprint=fp, evidence_file_hashes=per_file)
    else:
        rec.update(evidence_artifacts=None, evidence_fingerprint=None)
    write_settlement(vp.settlement_json, rec)
    log_decision(decision=f'v4b cache settlement after one rebuild: {final_decision}',
                 evidence_used=f'pre-rebuild reasons={reasons}; post-rebuild summary in cache_settlement.json',
                 host_visible=False, category='bug_fix', log_path=DECISION_LOG_PATH)
else:
    print('no rebuild executed in this run')

original C3 triples absent from rebuilt QCR (selection moved): 0


,metric,n_mismatch,median_abs_diff,p95_abs_diff,max_abs_diff
0,cosine,0,0.0,0.0,0.0
1,modified_cosine,0,0.0,0.0,0.0
2,peak_overlap_frac,0,0.0,0.0,0.0
3,neutral_loss_cosine,0,0.0,0.0,0.0


POST-REBUILD DECISION: REBUILT_AND_VALIDATED []


## 9. Persisted settlement decision

In [23]:
with open(vp.settlement_json, encoding='utf-8') as f:
    final_record = json.load(f)
fields = ['decision', 'n_pairs_checked', 'n_value_mismatches', 'n_tier_changes', 'n_threshold_crossings', 'max_abs_diff_cosine',
          'max_abs_diff_modified_cosine', 'max_abs_diff_peak_overlap', 'max_abs_diff_neutral_loss', 'top100_tie_cases',
          'top100_unstable_cases', 'aggregated_feature_mismatch_count', 'evidence_fingerprint']
display(pd.Series({k: final_record.get(k) for k in fields}, name='cache_settlement').to_frame())
if final_record['decision'] in ('ACCEPT_EXISTING', 'REBUILT_AND_VALIDATED'):
    load_settlement_or_refuse(vp.settlement_json)  # the exact gate 10v4b_01 applies
    print('\nSETTLED -> proceed to scripts/v4b_build_scale_features.py, then 10v4b_01_modeA_ranking_freeze.ipynb')
else:
    print(f"\nNOT SETTLED ({final_record['decision']}) -> 10v4b_01 will refuse to run")
print(f'\nruntime: {time.time() - NOTEBOOK_START:.1f}s')

,cache_settlement
decision,REBUILT_AND_VALIDATED
n_pairs_checked,1723
n_value_mismatches,0
n_tier_changes,0
n_threshold_crossings,0
max_abs_diff_cosine,0.0
max_abs_diff_modified_cosine,0.0
max_abs_diff_peak_overlap,0.0
max_abs_diff_neutral_loss,0.0
top100_tie_cases,0



SETTLED -> proceed to scripts/v4b_build_scale_features.py, then 10v4b_01_modeA_ranking_freeze.ipynb

runtime: 2485.5s
